<a href="https://colab.research.google.com/github/arman-hossain45/LSTM_Project/blob/main/lstm_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Simple lstm project
# next words prediction



In [77]:
!pip install nltk # natural language  precessing library

In [78]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter# count frequency of elements in a list or text
from torch.utils.data import Dataset,DataLoader
from nltk.tokenize import word_tokenize # splits text into individual words/tokens
import nltk # used for NLP tasks like tokenize ,stemming ,stopwords removal etc

In [79]:

documents = """The Last Train Journey

On a cold winter evening, Daniel arrived at the old railway station carrying a small leather bag.
The station was nearly empty except for a few passengers waiting quietly on wooden benches.
A loud whistle echoed through the air as the final train of the night slowly entered the platform.

Daniel checked the ticket in his pocket and walked toward the train.
The metal doors opened with a creaking sound and warm air rushed outside.
Inside the train, yellow lights flickered softly above rows of empty seats.

He chose a seat beside the window and placed his bag carefully near his feet.
As the train began to move, the city lights slowly disappeared behind thick clouds of fog.
Snow started falling gently outside while distant mountains appeared under the moonlight.

Across the aisle sat an elderly woman reading an old book with a blue cover.
Every few minutes she looked outside the window as if searching for something familiar.
Daniel noticed a silver necklace around her neck shaped like a small compass.

After several hours, the train stopped at a remote station surrounded by tall pine trees.
Very few people entered or left the train at that place.
A strange silence covered the station as the doors closed once again.

Curious about the woman, Daniel finally started a conversation.
The woman introduced herself as Eleanor and explained that she was returning to her hometown after many years.
She said the town was hidden deep within the northern mountains and was known for its beautiful frozen lake.

As the journey continued, Eleanor shared stories from her childhood.
She described colorful festivals, music performances, and lanterns floating across the lake during winter nights.
Daniel listened carefully while the train moved through snowy valleys and dark forests.

Near midnight, the train suddenly slowed down because heavy snow had covered the tracks ahead.
Passengers became nervous and began discussing possible delays.
Conductors moved through the cabins trying to calm everyone.

Daniel looked outside and saw workers removing snow under bright floodlights.
The freezing wind shook the train slightly while snow continued falling from the dark sky.
Despite the delay, Eleanor remained calm and continued telling her stories.

Several hours later, the tracks were finally cleared and the train resumed its journey.
The passengers sighed with relief as warm coffee was served throughout the cabins.
Daniel and Eleanor continued talking about travel, history, and forgotten places around the world.

Just before sunrise, the train reached the mountain town.
The station was small but beautifully decorated with glowing lamps and snow-covered rooftops.
People wearing thick winter coats walked slowly across the icy streets.

Before leaving, Eleanor handed Daniel a folded piece of paper.
Inside it was a drawing of the frozen lake along with a short message thanking him for the conversation.
Daniel smiled and carefully placed the paper inside his notebook.

As the train departed once again, Daniel stood on the platform watching the sunrise over the mountains.
The cold air, quiet streets, and distant church bells created a peaceful atmosphere he would never forget.
Years later, Daniel would still remember that mysterious winter journey and the stories shared by Eleanor on the last train of the night.
"""

In [80]:
documents

'The Last Train Journey\n\nOn a cold winter evening, Daniel arrived at the old railway station carrying a small leather bag.\nThe station was nearly empty except for a few passengers waiting quietly on wooden benches.\nA loud whistle echoed through the air as the final train of the night slowly entered the platform.\n\nDaniel checked the ticket in his pocket and walked toward the train.\nThe metal doors opened with a creaking sound and warm air rushed outside.\nInside the train, yellow lights flickered softly above rows of empty seats.\n\nHe chose a seat beside the window and placed his bag carefully near his feet.\nAs the train began to move, the city lights slowly disappeared behind thick clouds of fog.\nSnow started falling gently outside while distant mountains appeared under the moonlight.\n\nAcross the aisle sat an elderly woman reading an old book with a blue cover.\nEvery few minutes she looked outside the window as if searching for something familiar.\nDaniel noticed a silver 

In [81]:
# NLTK does not with pretrained tokenizers by default

# Tokenization
nltk.download('punkt')

# sosme newer NLTK versions require additional tokennizer tables.

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**Question: Why do we use nltk.download('punkt')?**

Answer:

punkt is a pre-trained tokenizer model used by NLTK for sentence and word tokenization. It helps identify sentence boundaries and split text into meaningful tokens.

# Tokenize


In [82]:

tokens = word_tokenize(documents.lower()) # convet full paragraph text into token
tokens

['the',
 'last',
 'train',
 'journey',
 'on',
 'a',
 'cold',
 'winter',
 'evening',
 ',',
 'daniel',
 'arrived',
 'at',
 'the',
 'old',
 'railway',
 'station',
 'carrying',
 'a',
 'small',
 'leather',
 'bag',
 '.',
 'the',
 'station',
 'was',
 'nearly',
 'empty',
 'except',
 'for',
 'a',
 'few',
 'passengers',
 'waiting',
 'quietly',
 'on',
 'wooden',
 'benches',
 '.',
 'a',
 'loud',
 'whistle',
 'echoed',
 'through',
 'the',
 'air',
 'as',
 'the',
 'final',
 'train',
 'of',
 'the',
 'night',
 'slowly',
 'entered',
 'the',
 'platform',
 '.',
 'daniel',
 'checked',
 'the',
 'ticket',
 'in',
 'his',
 'pocket',
 'and',
 'walked',
 'toward',
 'the',
 'train',
 '.',
 'the',
 'metal',
 'doors',
 'opened',
 'with',
 'a',
 'creaking',
 'sound',
 'and',
 'warm',
 'air',
 'rushed',
 'outside',
 '.',
 'inside',
 'the',
 'train',
 ',',
 'yellow',
 'lights',
 'flickered',
 'softly',
 'above',
 'rows',
 'of',
 'empty',
 'seats',
 '.',
 'he',
 'chose',
 'a',
 'seat',
 'beside',
 'the',
 'window',
 'a

# Build vocabs form

In [83]:

# count the words
Counter(tokens)

Counter({'the': 55,
         'last': 2,
         'train': 14,
         'journey': 4,
         'on': 4,
         'a': 16,
         'cold': 2,
         'winter': 4,
         'evening': 1,
         ',': 19,
         'daniel': 11,
         'arrived': 1,
         'at': 3,
         'old': 2,
         'railway': 1,
         'station': 5,
         'carrying': 1,
         'small': 3,
         'leather': 1,
         'bag': 2,
         '.': 39,
         'was': 7,
         'nearly': 1,
         'empty': 2,
         'except': 1,
         'for': 4,
         'few': 3,
         'passengers': 3,
         'waiting': 1,
         'quietly': 1,
         'wooden': 1,
         'benches': 1,
         'loud': 1,
         'whistle': 1,
         'echoed': 1,
         'through': 3,
         'air': 3,
         'as': 8,
         'final': 1,
         'of': 6,
         'night': 2,
         'slowly': 3,
         'entered': 2,
         'platform': 2,
         'checked': 1,
         'ticket': 1,
         'in': 1,
      

In [84]:
# found in this vocab unique vocab
Counter(tokens).keys()

dict_keys(['the', 'last', 'train', 'journey', 'on', 'a', 'cold', 'winter', 'evening', ',', 'daniel', 'arrived', 'at', 'old', 'railway', 'station', 'carrying', 'small', 'leather', 'bag', '.', 'was', 'nearly', 'empty', 'except', 'for', 'few', 'passengers', 'waiting', 'quietly', 'wooden', 'benches', 'loud', 'whistle', 'echoed', 'through', 'air', 'as', 'final', 'of', 'night', 'slowly', 'entered', 'platform', 'checked', 'ticket', 'in', 'his', 'pocket', 'and', 'walked', 'toward', 'metal', 'doors', 'opened', 'with', 'creaking', 'sound', 'warm', 'rushed', 'outside', 'inside', 'yellow', 'lights', 'flickered', 'softly', 'above', 'rows', 'seats', 'he', 'chose', 'seat', 'beside', 'window', 'placed', 'carefully', 'near', 'feet', 'began', 'to', 'move', 'city', 'disappeared', 'behind', 'thick', 'clouds', 'fog', 'snow', 'started', 'falling', 'gently', 'while', 'distant', 'mountains', 'appeared', 'under', 'moonlight', 'across', 'aisle', 'sat', 'an', 'elderly', 'woman', 'reading', 'book', 'blue', 'cover

In [85]:


# builld vocab

vocab = {'<UNK>':0}

In [86]:


# iterate the overall unique  vocab
for token in  Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)
vocab

# formed the vocabulary   and give in this unique number


{'<UNK>': 0,
 'the': 1,
 'last': 2,
 'train': 3,
 'journey': 4,
 'on': 5,
 'a': 6,
 'cold': 7,
 'winter': 8,
 'evening': 9,
 ',': 10,
 'daniel': 11,
 'arrived': 12,
 'at': 13,
 'old': 14,
 'railway': 15,
 'station': 16,
 'carrying': 17,
 'small': 18,
 'leather': 19,
 'bag': 20,
 '.': 21,
 'was': 22,
 'nearly': 23,
 'empty': 24,
 'except': 25,
 'for': 26,
 'few': 27,
 'passengers': 28,
 'waiting': 29,
 'quietly': 30,
 'wooden': 31,
 'benches': 32,
 'loud': 33,
 'whistle': 34,
 'echoed': 35,
 'through': 36,
 'air': 37,
 'as': 38,
 'final': 39,
 'of': 40,
 'night': 41,
 'slowly': 42,
 'entered': 43,
 'platform': 44,
 'checked': 45,
 'ticket': 46,
 'in': 47,
 'his': 48,
 'pocket': 49,
 'and': 50,
 'walked': 51,
 'toward': 52,
 'metal': 53,
 'doors': 54,
 'opened': 55,
 'with': 56,
 'creaking': 57,
 'sound': 58,
 'warm': 59,
 'rushed': 60,
 'outside': 61,
 'inside': 62,
 'yellow': 63,
 'lights': 64,
 'flickered': 65,
 'softly': 66,
 'above': 67,
 'rows': 68,
 'seats': 69,
 'he': 70,
 'chose

In [87]:
len(vocab)

284

# Extarct sentence from data


In [88]:
documents

'The Last Train Journey\n\nOn a cold winter evening, Daniel arrived at the old railway station carrying a small leather bag.\nThe station was nearly empty except for a few passengers waiting quietly on wooden benches.\nA loud whistle echoed through the air as the final train of the night slowly entered the platform.\n\nDaniel checked the ticket in his pocket and walked toward the train.\nThe metal doors opened with a creaking sound and warm air rushed outside.\nInside the train, yellow lights flickered softly above rows of empty seats.\n\nHe chose a seat beside the window and placed his bag carefully near his feet.\nAs the train began to move, the city lights slowly disappeared behind thick clouds of fog.\nSnow started falling gently outside while distant mountains appeared under the moonlight.\n\nAcross the aisle sat an elderly woman reading an old book with a blue cover.\nEvery few minutes she looked outside the window as if searching for something familiar.\nDaniel noticed a silver 

In [89]:
input_sentence = documents.split('\n')

In [90]:
input_sentence[3]

'The station was nearly empty except for a few passengers waiting quietly on wooden benches.'

In [91]:
def text_to_indices(sentence,vocab):
  numerical_sentence = []
  for token in sentence :
    if token in vocab :
      numerical_sentence.append(vocab[token])

    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [92]:
input_numerical_sentence = []
for sentence in input_sentence:
   input_numerical_sentence.append(text_to_indices(word_tokenize(sentence.lower()),vocab))


In [93]:
input_numerical_sentence
# we see that in a single dictionary every sentence is in list with dictionary formate

[[1, 2, 3, 4],
 [],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19, 20, 21],
 [1, 16, 22, 23, 24, 25, 26, 6, 27, 28, 29, 30, 5, 31, 32, 21],
 [6, 33, 34, 35, 36, 1, 37, 38, 1, 39, 3, 40, 1, 41, 42, 43, 1, 44, 21],
 [],
 [11, 45, 1, 46, 47, 48, 49, 50, 51, 52, 1, 3, 21],
 [1, 53, 54, 55, 56, 6, 57, 58, 50, 59, 37, 60, 61, 21],
 [62, 1, 3, 10, 63, 64, 65, 66, 67, 68, 40, 24, 69, 21],
 [],
 [70, 71, 6, 72, 73, 1, 74, 50, 75, 48, 20, 76, 77, 48, 78, 21],
 [38, 1, 3, 79, 80, 81, 10, 1, 82, 64, 42, 83, 84, 85, 86, 40, 87, 21],
 [88, 89, 90, 91, 61, 92, 93, 94, 95, 96, 1, 97, 21],
 [],
 [98, 1, 99, 100, 101, 102, 103, 104, 101, 14, 105, 56, 6, 106, 107, 21],
 [108, 27, 109, 110, 111, 61, 1, 74, 38, 112, 113, 26, 114, 115, 21],
 [11, 116, 6, 117, 118, 119, 120, 121, 122, 123, 6, 18, 124, 21],
 [],
 [125, 126, 127, 10, 1, 3, 128, 13, 6, 129, 16, 130, 131, 132, 133, 134, 21],
 [135, 27, 136, 43, 137, 138, 1, 3, 13, 139, 140, 21],
 [6, 141, 142, 143, 1, 16, 38, 1, 54, 144, 145, 146

we found every sentence as a unique number

# Form training sequence

In [94]:
training_sequences = []
for sentence in input_numerical_sentence:
  for i in range (1,len(sentence)):
    training_sequences.append(sentence[:i+1])



In [95]:
training_sequences

[[1, 2],
 [1, 2, 3],
 [1, 2, 3, 4],
 [5, 6],
 [5, 6, 7],
 [5, 6, 7, 8],
 [5, 6, 7, 8, 9],
 [5, 6, 7, 8, 9, 10],
 [5, 6, 7, 8, 9, 10, 11],
 [5, 6, 7, 8, 9, 10, 11, 12],
 [5, 6, 7, 8, 9, 10, 11, 12, 13],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19, 20],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19, 20, 21],
 [1, 16],
 [1, 16, 22],
 [1, 16, 22, 23],
 [1, 16, 22, 23, 24],
 [1, 16, 22, 23, 24, 25],
 [1, 16, 22, 23, 24, 25, 26],
 [1, 16, 22, 23, 24, 25, 26, 6],
 [1, 16, 22, 23, 24, 25, 26, 6, 27],
 [1, 16, 22, 23, 24, 25, 26, 6, 27, 28],
 [1, 16, 22, 23, 24, 2

In [75]:
len(training_sequences)

555

In [96]:
training_sequences[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [5, 6], [5, 6, 7]]

In [74]:
# we found the highest big sequence

len_list = []

for sequence in training_sequences:
  len_list.append(len(sequence))



In [76]:
max(len_list)

25

Here our main len is 25 so for that reason when we found 2 len then we fillup this 23 values with 0 to fullfill the target 25 so for that reason we written this below formula that i give in below

In [99]:
# now we fill up this with 0 values which is not as fillup this full length
[0]*(max(len_list)-len(sequence))


training_sequences[0]

[1, 2]

In [101]:

padded_training_sequences = []
for sequence in training_sequences:
  padded_training_sequences.append([0]*(max(len_list)-len(sequence))+sequence)


In [102]:
padded_training_sequences

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7, 8],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7, 8, 9],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7, 8, 9, 10],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7, 8, 9, 10, 11],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
 

we see that every padding is same shape which is 25

In [103]:
padded_training_sequences[5]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7, 8]

In [104]:
padded_training_sequences[4]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 7]

In [105]:
padded_training_sequences[2]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4]

In [106]:
padded_training_sequences[1]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3]

# tensor convert it all this data

and train test split

In [107]:
padded_training_sequences = torch.tensor(padded_training_sequences,dtype=torch.long)
padded_training_sequences

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0, 158,  ...,   3,  40,   1],
        [  0, 158, 223,  ...,  40,   1,  41],
        [158, 223,  10,  ...,   1,  41,  21]])

In [108]:
x = padded_training_sequences[:,:-1]# last ar column gula nibo na
x

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0, 158,  ...,   2,   3,  40],
        [  0, 158, 223,  ...,   3,  40,   1],
        [158, 223,  10,  ...,  40,   1,  41]])

In [109]:
y = padded_training_sequences[:,-1]
y

tensor([  2,   3,   4,   6,   7,   8,   9,  10,  11,  12,  13,   1,  14,  15,
         16,  17,   6,  18,  19,  20,  21,  16,  22,  23,  24,  25,  26,   6,
         27,  28,  29,  30,   5,  31,  32,  21,  33,  34,  35,  36,   1,  37,
         38,   1,  39,   3,  40,   1,  41,  42,  43,   1,  44,  21,  45,   1,
         46,  47,  48,  49,  50,  51,  52,   1,   3,  21,  53,  54,  55,  56,
          6,  57,  58,  50,  59,  37,  60,  61,  21,   1,   3,  10,  63,  64,
         65,  66,  67,  68,  40,  24,  69,  21,  71,   6,  72,  73,   1,  74,
         50,  75,  48,  20,  76,  77,  48,  78,  21,   1,   3,  79,  80,  81,
         10,   1,  82,  64,  42,  83,  84,  85,  86,  40,  87,  21,  89,  90,
         91,  61,  92,  93,  94,  95,  96,   1,  97,  21,   1,  99, 100, 101,
        102, 103, 104, 101,  14, 105,  56,   6, 106, 107,  21,  27, 109, 110,
        111,  61,   1,  74,  38, 112, 113,  26, 114, 115,  21, 116,   6, 117,
        118, 119, 120, 121, 122, 123,   6,  18, 124,  21, 126, 1

In [110]:
len(y)

555

# Custom data set and data loader

In [114]:
class CustomDataset(Dataset):
  def __init__(self,x,y):
     self.x = x
     self.y =y
  def __len__(self):
     return self.x.shape[0]

  def __getitem__(self,index):
    return self.x[index],self.y[index]

In [115]:
dataset = CustomDataset(x,y)

In [116]:
len(dataset)

555

In [117]:
# data loader objecct
dataloader =DataLoader(dataset,batch_size=32,shuffle =True)

In [118]:
for input , output in dataloader :
  print(input,output)

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0, 204, 185,  36],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   6,  33,  34,  35,  36],
        [  0,   0,   0,   0,   0,   0,   0,   6,  33,  34,  35,  36,   1,  37,
          38,   1,  39,   3,  40,   1,  41,  42,  43,   1],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0, 239, 254],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,  88,  89,  90,  91],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  11,
         184,  76,  92,   1,   3, 185,  36, 186, 187,  50],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  11, 184,
          76,  92,   1,   3, 185,  36, 186, 187,  50, 188],
        [  0,   0,   0,   0

In [119]:
x.shape

torch.Size([555, 24])

In [120]:
y.shape

torch.Size([555])

# LSTM Training and implementation